# Thư viện và Dữ liệu

In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import ParameterGrid
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
# tải dữ liệu
# Mỗi hàng = 1 khách hàng × 1 tháng
df = pd.read_csv('../tnbike_data_customers.csv', index_col=0, parse_dates=True)
df.head()

In [ ]:
# đổi tên biến
df = df.rename({'bought': 'y'}, axis=1)
df

# Chuẩn bị cho GBM

In [ ]:
# đặc trưng và nhãn
feature_cols = [c for c in df.columns if c != 'y']
X = df[feature_cols]
y = df['y']

In [ ]:
# chia tập train / tập dự báo (Q2/2026)
training_X  = X[X.index <  '2026-04-01']
training_y  = y[y.index <  '2026-04-01']
future_X    = X[X.index >= '2026-04-01']

In [ ]:
# scale đặc trưng
scaler   = StandardScaler()
train_X_scaled  = scaler.fit_transform(training_X)
future_X_scaled = scaler.transform(future_X)

In [ ]:
# scale nhãn (giữ nguyên)
print(f"Train shape: {train_X_scaled.shape}")
print(f"Future shape: {future_X_scaled.shape}")

# Mô hình GBM Classifier

In [ ]:
# lấy tham số tốt nhất
parameters = pd.read_csv("../Forecasting Product/best_params_classifier.csv", index_col=0)
parameters

In [ ]:
# trích xuất giá trị tham số
n_estimators    = int(parameters.loc["n_estimators"][0])
max_depth       = int(parameters.loc["max_depth"][0])
learning_rate   = float(parameters.loc["learning_rate"][0])
subsample       = float(parameters.loc["subsample"][0])
colsample_bytree= float(parameters.loc["colsample_bytree"][0])

In [ ]:
# mô hình GBM
model = XGBClassifier(n_estimators=n_estimators,
                      max_depth=max_depth,
                      learning_rate=learning_rate,
                      subsample=subsample,
                      colsample_bytree=colsample_bytree,
                      use_label_encoder=False,
                      eval_metric='logloss',
                      random_state=42)
model.fit(train_X_scaled, training_y)

# Dự báo

In [ ]:
# dự báo xác suất mua hàng Q2/2026
predictions_proba = model.predict_proba(future_X_scaled)[:, 1]
predictions_gbm   = pd.Series(predictions_proba,
                               index=future_X.index,
                               name="gbm_proba")

In [ ]:
# trực quan hóa
predictions_gbm.plot(kind='bar', figsize=(15, 6),
                     title='Xác suất mua hàng Q2/2026 theo đại lý',
                     legend=True);

In [ ]:
# xuất kết quả
predictions_gbm.to_csv("Ensemble/predictions_gbm.csv")